In [ ]:
import cedalion.dot
import cedalion.io
import cedalion.nirs
import cedalion.vis.anatomy
import cedalion.vis.blocks as vbx
import pyvista as pv
from cedalion.geometry.landmarks import normalize_landmarks_labels
import cedalion.data
import numpy as np

In [ ]:
rec = cedalion.data.get_lumo_testdataset()

The dataset contains all SxD channel combinations. Restrict them to channels with S-D-distances below 3.5 cm

In [ ]:
ch_mask = cedalion.nirs.channel_distances(rec["amp"], rec.geo3d) < 3.5 * cedalion.units.cm

In [ ]:
amp = rec["amp"].sel(channel=ch_mask)
amp

In [ ]:
cedalion.vis.anatomy.plot_montage3D(amp, rec.geo3d)

In [ ]:
head_ijk = cedalion.dot.get_standard_headmodel("colin27")
head_ras = head_ijk.apply_transform(head_ijk.t_ijk2ras)

In [ ]:
geo3d = normalize_landmarks_labels(rec.geo3d)
geo3d_snapped = head_ras.align_and_snap_to_scalp(geo3d)

In [ ]:
Adot = cedalion.data.get_precomputed_sensitivity("lumo_testdataset", "colin27")

In [ ]:
sensitivity = Adot.sel(vertex=Adot.is_brain, wavelength=850).max("channel")

In [ ]:
plt = pv.Plotter()
vbx.plot_surface(plt, head_ras.scalp, color="w", opacity=0.5)
vbx.plot_surface(plt, head_ras.brain, color=np.log10(np.clip(sensitivity, 1e-3, 1e1)), cmap="jet")
vbx.plot_labeled_points(plt, geo3d_snapped)
plt.show()